# Ensemble Classification: Cloud APIs, Local Models, and Temperature Variants

CatLLM's ensemble mode runs multiple models and combines their classifications with consensus voting. This notebook covers every ensemble pattern — from multi-provider cloud ensembles to temperature sweeps to hybrid local+cloud setups.

**What this notebook covers:**
- How ensemble consensus works (voting, agreement scores, thresholds)
- Multi-provider cloud ensemble (parallel execution)
- Temperature ensembles (same model at different creativity settings)
- Local-only Ollama ensemble (sequential execution)
- Hybrid cloud + local ensemble
- How CatLLM handles parallel vs sequential execution and why

**What you need:**
- Python 3.9+
- `pip install catllm`
- API keys for at least one cloud provider (OpenAI, Anthropic, Google, etc.)
- For local model examples: [Ollama](https://ollama.com/download) installed and running

---

## How Ensemble Classification Works

When you pass a `models` list with multiple entries, CatLLM:

1. **Sends each text to every model** in the list
2. **Collects binary 0/1 votes** from each model for each category
3. **Applies consensus voting** — a category is marked "1" only if the fraction of models voting "1" meets your `consensus_threshold`
4. **Records agreement scores** — the fraction of models that agreed with the final consensus decision

**Example** (3 models, 1 category):
```
Model A votes: 1    Model B votes: 1    Model C votes: 0
Positive rate: 2/3 = 0.667

consensus_threshold="majority"  (0.50) → consensus = 1, agreement = 0.667
consensus_threshold="two-thirds" (0.67) → consensus = 1, agreement = 0.667
consensus_threshold="unanimous" (1.00) → consensus = 0, agreement = 0.333
```

**Why ensembles help:** Different models have different biases. Where one model over-classifies, another may be conservative. Unanimous consensus aggressively filters false positives — empirically, this produces the highest accuracy.

In [ ]:
import catllm as cat
import pandas as pd

## Sample Data

In [ ]:
responses = [
    "because i dont like living here",
    "for a bigger house",
    "to be with my wife",
    "got a new job in another state",
    "rent was too expensive so we had to leave",
]

categories = [
    "to start living with or to stay with partner/spouse",
    "relationship change (divorce, breakup, etc)",
    "the person had a job or school or career change, including transferred and retired",
    "financial reasons (rent is too expensive, pay raise, etc)",
    "related specifically to features of the home, such as a bigger or smaller yard",
]

## Cloud Ensemble: Multiple Providers

The most common ensemble pattern — run 2+ cloud models from different providers. Each model processes all rows **in parallel** using a thread pool, so total wall-clock time is roughly the time of the slowest model.

Pass a `models` list of `(model, provider, api_key)` tuples.

In [ ]:
result = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
    ],
    consensus_threshold="unanimous",
)

result.head()

## Understanding Ensemble Output

The returned DataFrame includes:

| Column pattern | Meaning |
|---------------|----------|
| `survey_input` | The original text |
| `category_1_gpt_4o_mini` | Model A's vote for category 1 (0 or 1) |
| `category_1_claude_sonnet_...` | Model B's vote for category 1 |
| `category_1_consensus` | Final consensus decision (0 or 1) |
| `category_1_agreement` | Fraction of models matching consensus (0.0–1.0) |
| `processing_status` | `"success"`, `"partial"`, `"error"`, or `"skipped"` |
| `failed_models` | Comma-separated model names that failed (if any) |

Low agreement scores (e.g., 0.5) flag rows where models disagreed — these may need manual review.

In [ ]:
# View consensus columns only
consensus_cols = [c for c in result.columns if "consensus" in c or "agreement" in c]
result[["survey_input"] + consensus_cols].head()

## Consensus Thresholds

The `consensus_threshold` parameter controls how strict the voting is:

| Threshold | Value | Effect |
|-----------|-------|--------|
| `"unanimous"` | 1.0 | **All** models must agree. Highest precision, fewest false positives. **Default.** |
| `"two-thirds"` | 0.67 | Two-thirds of models must agree. Balanced. |
| `"majority"` | 0.5 | Half of models must agree. Highest recall, more false positives. |
| `0.75` | 0.75 | Custom numeric threshold (any float 0–1). |

**Recommendation:** Start with `"unanimous"` (the default). It aggressively eliminates false positives and empirically produces the highest accuracy.

In [ ]:
# More permissive threshold — useful with 3+ models
result_majority = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
        ("gemini-2.0-flash", "google", "YOUR_GOOGLE_KEY"),
    ],
    consensus_threshold="majority",
)

result_majority.head()

## Temperature Ensembles

Instead of using different models, you can ensemble the **same model at different temperatures** (creativity settings). Lower temperatures produce more deterministic output; higher temperatures introduce more variation. Consensus voting across temperatures filters out random noise while preserving confident classifications.

Use the **4-tuple format** to set per-model creativity: `(model, provider, api_key, {"creativity": value})`

Column names automatically include the temperature suffix (e.g., `_t25` for creativity=0.25).

In [ ]:
result_temp = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.0}),
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.3}),
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.7}),
    ],
    consensus_threshold="majority",
)

# Output columns include temperature suffixes:
# category_1_gpt_4o_mini_t0, category_1_gpt_4o_mini_t30, category_1_gpt_4o_mini_t70
# category_1_consensus, category_1_agreement
result_temp.head()

## Combining Temperature + Model Diversity

For maximum robustness, combine different models at different temperatures. This captures both model-level and temperature-level diversity.

In [ ]:
result_diverse = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        # GPT at two temperatures
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.0}),
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.5}),
        # Claude at two temperatures
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY", {"creativity": 0.0}),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY", {"creativity": 0.5}),
    ],
    consensus_threshold="majority",
)

result_diverse.head()

## How Execution Works: Parallel vs Sequential

CatLLM automatically chooses how to run your ensemble based on **where the models live**:

### Cloud Models → Parallel Execution

When your ensemble uses cloud APIs (OpenAI, Anthropic, Google, etc.), all models process each row **simultaneously** using a thread pool. Each model sends its HTTP request at the same time, and results are collected as they arrive.

```
Row 1 → [GPT ───────── done]  Wall-clock time ≈
         [Claude ────── done]  max(GPT, Claude, Gemini)
         [Gemini ────── done]
```

This works because each API call is independent — they run on different servers with no shared resources.

### Ollama Models → Sequential Execution

When **all** models are Ollama (local), CatLLM automatically switches to **sequential** mode — processing one model at a time. This is intentional:

```
Row 1 → [Llama ──── done][Qwen ──── done][Mistral ──── done]
         Wall-clock time = sum(Llama, Qwen, Mistral)
```

**Why sequential?** All Ollama models share the same CPU/GPU. Running them in parallel forces the hardware to context-switch between models, thrashing caches and memory. Sequential execution lets each model use the full hardware, which is faster overall despite processing one at a time.

### Mixed (Cloud + Ollama) → Parallel with Auto-Detection

When the ensemble mixes cloud and local models, CatLLM runs in **parallel mode** — cloud models fire HTTP requests while the Ollama model runs locally. The Ollama model naturally serializes on the local hardware while cloud requests happen concurrently.

### Override with `parallel`

You can force the execution mode:
- `parallel=True` — Force parallel (useful if running Ollama on a multi-GPU server)
- `parallel=False` — Force sequential (useful for debugging)

In [ ]:
# This runs all cloud models in parallel automatically
result_parallel = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
    ],
    # parallel=True is auto-detected for cloud models
)

result_parallel.head()

## Local-Only Ollama Ensemble

Run multiple local models with consensus voting. CatLLM automatically switches to sequential execution when all models are Ollama.

**Important:** Since Ollama models share the same hardware, each model runs one at a time per row. An ensemble of 3 local models takes roughly 3x the time of a single model — but the accuracy gain from consensus voting is often worth it.

Set `model_source="ollama"` in each tuple. The API key is required by the format but ignored.

In [ ]:
result_local = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("llama3.2", "ollama", "not-needed"),
        ("qwen2.5:7b", "ollama", "not-needed"),
    ],
    consensus_threshold="unanimous",
    auto_download=True,
    # parallel is auto-set to False (sequential) for all-Ollama ensembles
)

result_local.head()

## Local Temperature Ensemble

Temperature ensembles work with Ollama too. This is an effective way to improve accuracy with a single local model — run it 3 times at different temperatures and vote.

Since all calls go to the same local model, execution is always sequential.

In [ ]:
result_local_temp = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("llama3.2", "ollama", "not-needed", {"creativity": 0.0}),
        ("llama3.2", "ollama", "not-needed", {"creativity": 0.3}),
        ("llama3.2", "ollama", "not-needed", {"creativity": 0.7}),
    ],
    consensus_threshold="majority",
)

result_local_temp.head()

## Hybrid Ensemble: Cloud + Local Models

Combine the strengths of cloud and local models in one ensemble. Cloud models are generally more accurate; local models add diversity and reduce cloud API costs.

In a hybrid ensemble:
- Cloud models send HTTP requests **in parallel** (no shared resources)
- The Ollama model runs **locally** on your machine at the same time
- CatLLM uses parallel mode since not all models are Ollama

```
Row 1 → [GPT ──────── done]       Cloud: parallel HTTP
         [Claude ───── done]
         [Llama (local) ── done]   Local: runs on your CPU/GPU
```

In [ ]:
result_hybrid = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
        ("llama3.2", "ollama", "not-needed"),
    ],
    consensus_threshold="majority",  # 2/3 must agree
    auto_download=True,
)

result_hybrid.head()

## Hybrid + Temperature Ensemble

The most comprehensive ensemble pattern: multiple providers, multiple temperatures. This maximizes diversity at the cost of more API calls.

In [ ]:
result_full = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        # Cloud models at two temperatures
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.0}),
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY", {"creativity": 0.5}),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY", {"creativity": 0.0}),
        # Local model
        ("llama3.2", "ollama", "not-needed", {"creativity": 0.0}),
    ],
    consensus_threshold="majority",
    auto_download=True,
)

result_full.head()

## Robustness Features for Ensembles

Ensembles involve more API calls, so robustness matters more:

- **`safety=True`**: Save progress after each row — critical for large ensemble runs
- **`fail_strategy="partial"`** (default): If one model fails, use the remaining models' votes. `"strict"` blanks the entire row.
- **`batch_retries`**: Retry failed (row, model) pairs after the main pass
- **`row_delay`**: Pause between rows — useful when multiple models share the same API key/rate limit

In [ ]:
result_robust = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
    ],
    consensus_threshold="unanimous",
    safety=True,
    filename="ensemble_results.csv",
    save_directory="./output",
    fail_strategy="partial",
    batch_retries=2,
    row_delay=0.5,
)

result_robust.head()

## Batch Mode Ensemble (50% Cost Savings)

For large datasets, `batch_mode=True` submits each model's work as a separate async batch job. All models' batch jobs run **concurrently** on the provider side.

- Supported providers: OpenAI, Anthropic, Google, Mistral, xAI
- Unsupported providers (HuggingFace, Perplexity, Ollama) **automatically fall back** to synchronous calls and are merged with the batch results
- Text input only (no PDF/image)

In [ ]:
result_batch = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
    ],
    consensus_threshold="unanimous",
    batch_mode=True,
)

result_batch.head()

## Embedding Tiebreaker for Consensus Ties

When consensus voting produces a **true tie** — exactly half the models vote 1 and half vote 0 — the default behavior is to mark the category as 0. The **embedding tiebreaker** provides a smarter resolution by comparing the tied text to known examples.

**How it works:**

1. After classification, CatLLM identifies rows where models are evenly split (e.g., 2 vote yes, 2 vote no)
2. It builds **centroids** (average embeddings) for each category using rows where all models unanimously agreed
3. For tied rows, it computes the embedding similarity to both the "positive" and "negative" centroids
4. The tie is broken in favor of whichever centroid the text is closer to

**Requirements:**
- `pip install catllm[embeddings]` — this installs the `sentence-transformers` library
- On first use, CatLLM automatically downloads the [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5) embedding model (~130MB) from HuggingFace Hub. This is a one-time download that gets cached locally for future runs. CatLLM will prompt you for confirmation before downloading.
- Multi-model ensemble only (ties can't happen with a single model)
- Text input only (not supported for PDF/image)
- Not supported in `batch_mode`
- Needs at least `min_centroid_size` (default 3) unanimously-agreed rows per category to build centroids

The output adds `category_N_resolved_by` columns showing whether each classification came from `"consensus"` or `"tiebreaker"`.

In [ ]:
result_tiebreaker = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "YOUR_ANTHROPIC_KEY"),
        ("gemini-2.0-flash", "google", "YOUR_GOOGLE_KEY"),
        ("mistral-small-latest", "mistral", "YOUR_MISTRAL_KEY"),
    ],
    consensus_threshold="majority",
    embedding_tiebreaker=True,
    min_centroid_size=3,  # Need at least 3 unanimous rows to build centroids
)

# Check which rows were resolved by the tiebreaker
resolved_cols = [c for c in result_tiebreaker.columns if "resolved_by" in c]
result_tiebreaker[["survey_input"] + resolved_cols].head()

## Quick Reference

### Ensemble Patterns

| Pattern | Execution | Cost | Accuracy |
|---------|-----------|------|----------|
| Multi-provider cloud | Parallel | Higher (N API calls per row) | Highest |
| Temperature sweep (cloud) | Parallel | Higher (N calls, same provider) | Good |
| Multi-model local | Sequential | Free (local inference) | Good |
| Temperature sweep (local) | Sequential | Free | Moderate |
| Hybrid (cloud + local) | Parallel | Medium | High |
| Batch ensemble (cloud) | Concurrent batch jobs | 50% cheaper | Same as cloud |

### Execution Mode

| Scenario | `parallel` auto-value | Why |
|----------|----------------------|-----|
| All cloud models | `True` | Independent servers, no shared resources |
| All Ollama models | `False` | Shared CPU/GPU — sequential avoids thrashing |
| Mixed cloud + Ollama | `True` | Cloud runs in parallel; Ollama serializes locally |

### Model Tuple Formats

```python
# 3-tuple: uses global creativity setting
("gpt-4o-mini", "openai", "sk-...")

# 4-tuple: per-model creativity override
("gpt-4o-mini", "openai", "sk-...", {"creativity": 0.5})

# Ollama: api_key is required but ignored
("llama3.2", "ollama", "not-needed")
```

### Consensus Thresholds

| Threshold | Value | Best for |
|-----------|-------|----------|
| `"unanimous"` | 1.0 | Maximizing precision (default) |
| `"two-thirds"` | 0.67 | Balanced precision/recall with 3+ models |
| `"majority"` | 0.5 | Maximizing recall |
| Custom float | 0.0–1.0 | Fine-tuning threshold to your data |

### Embedding Tiebreaker

| Parameter | Default | Effect |
|-----------|---------|--------|
| `embedding_tiebreaker` | `False` | Enable centroid-based tie resolution |
| `min_centroid_size` | `3` | Minimum unanimous rows needed to build centroids |

**Limitations:** Text input only, multi-model ensemble only, not supported in `batch_mode`.